In [ ]:
# Install required libraries
%pip install opencv-python
%pip install mediapipe
%pip install pyautogui
%pip install pynput

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Import core computer vision and GUI automation libraries
import cv2
import numpy as np
import util
import random
import mediapipe as mp
from mediapipe.tasks import python
import pyautogui
from pynput.mouse import Button, Controller
from mediapipe.tasks.python import vision

In [ ]:
# Initialize screen dimensions and mouse controller
screen_width, screen_height = pyautogui.size()
mouse = Controller()

# Initialize MediaPipe HandLandmarker Task for real-time video stream processing
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,  # Optimized for continuous video streams
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.7
)

# Create detector instance
detector = vision.HandLandmarker.create_from_options(options)

In [ ]:
def find_finger_tip(processed):
    """
    Extracts the normalized coordinates of the index finger tip from detector results.
    Landmark 8 corresponds to the index finger tip in MediaPipe's hand model.
    """
    if processed.hand_landmarks:
        hand_landmarks = processed.hand_landmarks[0]
        return hand_landmarks[8]  # Index 8 is INDEX_FINGER_TIP landmark

    return None

In [ ]:
def move_mouse(index_finger_tip):
    """
    Maps normalized finger tip coordinates (0.0 to 1.0) to screen resolution 
    and updates the system cursor position using PyAutoGUI.
    """
    if index_finger_tip is not None:
        x = int(index_finger_tip.x * screen_width)
        y = int(index_finger_tip.y * screen_height)
        pyautogui.moveTo(x, y)

In [ ]:
# Gesture condition evaluation functions using joint angles and landmark distances

def is_left_click(landmarks_list, thumb_index_distance):
    """Triggers Left Click based on index finger flexion and thumb separation."""
    return (util.get_angle(landmarks_list[5], landmarks_list[6], landmarks_list[8]) < 50 and 
            util.get_angle(landmarks_list[9], landmarks_list[10], landmarks_list[12]) > 90 and 
            thumb_index_distance > 50)

def is_right_click(landmarks_list, thumb_index_distance):
    """Triggers Right Click based on middle finger flexion and thumb separation."""
    return (util.get_angle(landmarks_list[5], landmarks_list[6], landmarks_list[8]) > 90 and 
            util.get_angle(landmarks_list[9], landmarks_list[10], landmarks_list[12]) < 50 and 
            thumb_index_distance > 50)

def is_double_click(landmarks_list, thumb_index_distance):
    """Triggers Double Click when index and middle fingers are simultaneously flexed."""
    return (util.get_angle(landmarks_list[5], landmarks_list[6], landmarks_list[8]) < 50 and 
            util.get_angle(landmarks_list[9], landmarks_list[10], landmarks_list[12]) < 50 and 
            thumb_index_distance > 50)

def is_screenshot(landmarks_list, thumb_index_distance):
    """Triggers a Screenshot action when hand gesture criteria are met."""
    return (util.get_angle(landmarks_list[5], landmarks_list[6], landmarks_list[8]) < 50 and 
            util.get_angle(landmarks_list[9], landmarks_list[10], landmarks_list[12]) > 90 and 
            util.get_angle(landmarks_list[13], landmarks_list[14], landmarks_list[16]) > 90 and 
            util.get_angle(landmarks_list[17], landmarks_list[18], landmarks_list[20]) > 90 and
            thumb_index_distance < 50)

In [ ]:
def detect_gestures(frame, landmarks_list, processed):
    """
    Evaluates detected hand landmarks, identifies gestures, and executes
    corresponding mouse/keyboard actions while displaying visual feedback on the frame.
    """
    if len(landmarks_list) >= 21:
        index_finger_tip = find_finger_tip(processed)
        thumb_index_distance = util.get_distance([landmarks_list[4], landmarks_list[5]])

        # Movement gesture condition
        if thumb_index_distance < 50 and util.get_angle(landmarks_list[5], landmarks_list[6], landmarks_list[8]) > 90:
            move_mouse(index_finger_tip)

        # Mouse click & screenshot gestures
        elif is_left_click(landmarks_list, thumb_index_distance):
            mouse.press(Button.left)
            mouse.release(Button.left)
            cv2.putText(frame, "Left Click", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        elif is_right_click(landmarks_list, thumb_index_distance):
            mouse.press(Button.right)
            mouse.release(Button.right)
            cv2.putText(frame, "Right Click", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        elif is_double_click(landmarks_list, thumb_index_distance):
            pyautogui.doubleClick()
            cv2.putText(frame, "Double Click", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        elif is_screenshot(landmarks_list, thumb_index_distance):        
            im1 = pyautogui.screenshot()
            label = random.randint(1, 1000)
            im1.save(f'screenshot_by_joo_mouse_{label}.png')
            cv2.putText(frame, "Screenshot Taken", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

In [ ]:
def main():
    """Main execution loop for video capture, MediaPipe inference, and UI rendering."""
    cap = cv2.VideoCapture(0)

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Mirror frame horizontally for intuitive self-tracking
            frame = cv2.flip(frame, 1)
            frameRGB = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Convert frame to MediaPipe Image format
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frameRGB)
            timestamp_ms = int(cv2.getTickCount() / cv2.getTickFrequency() * 1000)

            # Perform hand detection for video stream
            processed = detector.detect_for_video(mp_image, timestamp_ms)
            landmarks_list = []

            # Process hand landmarks if present
            if processed.hand_landmarks:
                hand_landmarks = processed.hand_landmarks[0]

                for lm in hand_landmarks:
                    landmarks_list.append((lm.x, lm.y))

                # Draw landmark circles on OpenCV output frame
                h, w, _ = frame.shape
                for lm in hand_landmarks:
                    cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 5, (0, 255, 0), -1)

                print("Landmarks:", landmarks_list)

            # Run gesture classification logic
            detect_gestures(frame, landmarks_list, processed)

            # Display feed
            cv2.imshow('Joo Mouse', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    finally:
        # Clean up video capture resources and close open windows
        cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Landmarks: [(0.590239405632019, 0.9325687885284424), (0.5787439346313477, 0.8920185565948486), (0.5579623579978943, 0.8649442195892334), (0.5341035723686218, 0.8537397384643555), (0.5116761922836304, 0.8488447666168213), (0.5761879682540894, 0.8703665137290955), (0.5453816056251526, 0.8602495193481445), (0.5194628834724426, 0.8572185635566711), (0.49724769592285156, 0.8571158647537231), (0.5788062810897827, 0.9020888209342957), (0.5280696153640747, 0.8970183730125427), (0.4939254820346832, 0.8954414129257202), (0.46903425455093384, 0.8958263993263245), (0.5751833319664001, 0.9318766593933105), (0.5256461501121521, 0.9275733232498169), (0.502094030380249, 0.9214077591896057), (0.48893651366233826, 0.916079044342041), (0.5673537254333496, 0.9576030969619751), (0.5293117761611938, 0.9503517150878906), (0.5127279162406921, 0.9414653182029724), (0.5037404894828796, 0.9346610903739929)]
Landmarks: [(0.5841245055198669, 0.9302078485488892), (0.5715146064758301, 0.8951172232627869), (0.5536993